In [1]:
import torch as th
from pprint import pprint

In [11]:
output_fp_3 = "/h/adamo/frag-gnn/inference_outputs/debug_iceberg_3/scaffold/iceberg_inten_100/s0/test.pkl"

In [13]:
output_3 = th.load(output_fp_3)

In [73]:
print(output_3["cos_sim_0.01"].mean(), output_3["cos_hun"].mean(), output_3["primary_loss"].mean())

tensor(0.6277) tensor(0.5979) tensor(0.3344)


In [17]:
output_fp_4 = "/h/adamo/frag-gnn/inference_outputs/debug_iceberg_4/scaffold/iceberg_inten_100/s0/test.pkl"

In [20]:
output_4 = th.load(output_fp_4)

In [74]:
print(output_4["cos_sim_0.01"].mean(), output_4["cos_hun"].mean(), output_4["primary_loss"].mean())

tensor(0.6199) tensor(0.5895) tensor(0.3344)


In [63]:
from fragnnet.utils.spec_utils import calculate_cosine_similarity, batched_bin_func, get_ints_transform_func, get_ints_untransform_func

mz_bin_res = 0.01
mz_max = 1500.0
sum_ints = True
agg = "sum"

def cosine_distance_fn(
    true_mzs,
    true_logprobs,
    true_batch_idxs,
    pred_mzs,
    pred_logprobs,
    pred_batch_idxs,
    **kwargs):
    cd = 1. - calculate_cosine_similarity(
        true_mzs,
        th.exp(true_logprobs),
        true_batch_idxs,
        pred_mzs,
        th.exp(pred_logprobs),
        pred_batch_idxs,
        mz_bin_res=mz_bin_res,
        mz_max=mz_max,
        sum_ints=sum_ints
    )
    return cd

ints_transform_fn = get_ints_transform_func("sqrt")
ints_untransform_fn = get_ints_untransform_func("sqrt")

In [24]:
list(output_3.keys())

['unique_id',
 'smiles',
 'mean_loss',
 'spec_per_group',
 'spec_per_mol',
 'group_per_mol',
 'true_mzs',
 'true_logprobs',
 'true_batch_idxs',
 'pred_mzs',
 'pred_logprobs',
 'pred_batch_idxs',
 'pred_formula_vecs',
 'pred_formula_logprobs',
 'pred_formula_batch_idxs',
 'loss',
 'primary_loss',
 'cos_hun',
 'true_oos_e',
 'cos_sim_0.01',
 'true_oos_prob']

In [64]:
def get_stats_d(output_d):
    pred_mzs = output_d["pred_mzs"]
    pred_logprobs = output_d["pred_logprobs"]
    pred_ints = output_d["pred_logprobs"].exp()
    pred_batch_idxs = output_d["pred_batch_idxs"]
    true_mzs = output_d["true_mzs"]
    true_logprobs = output_d["true_logprobs"]
    true_ints = output_d["true_logprobs"].exp()
    true_batch_idxs = output_d["true_batch_idxs"]
    # untransform
    pred_ints = ints_untransform_fn(
        pred_ints,
        pred_batch_idxs
    )
    true_ints = ints_untransform_fn(
        true_ints,
        true_batch_idxs
    )
    # bin
    pred_bin_idxs, pred_bin_ints, pred_bin_batch_idxs = batched_bin_func(
        pred_mzs,
        ints_transform_fn(pred_ints),
        pred_batch_idxs,
        mz_max=mz_max,
        mz_bin_res=mz_bin_res,
        agg=agg,
        sparse=True,
        remove_prec_peaks=False,
        prec_mzs=None
    )
    # calculate cosine distance
    cos_dist = cosine_distance_fn(
        true_mzs,
        th.log(true_ints),
        true_batch_idxs,
        pred_mzs,
        th.log(pred_ints),
        pred_batch_idxs,
    )
    d = {
        "pred_bin_ints": pred_bin_ints,
        "pred_bin_idxs": pred_bin_idxs,
        "pred_bin_batch_idxs": pred_bin_batch_idxs,
        "cos_dist": cos_dist
    }
    return d

In [65]:
stats_d_3 = get_stats_d(output_3)
stats_d_4 = get_stats_d(output_4)

In [66]:
1.-stats_d_3["cos_dist"].mean().item(), 1.-stats_d_4["cos_dist"].mean().item()

(0.6271163821220398, 0.6194114089012146)

In [67]:
stats_d_3["cos_dist"].shape

torch.Size([3341])

In [68]:
stats_d_4["cos_dist"].shape

torch.Size([3341])

In [69]:
stats_d_3["pred_bin_ints"].shape

torch.Size([1100859])

In [70]:
stats_d_4["pred_bin_ints"].shape

torch.Size([1100859])

In [71]:
(stats_d_3["pred_bin_ints"] - stats_d_4["pred_bin_ints"]).abs().mean()

tensor(0.0096)

In [72]:
stats_d_3["pred_bin_ints"].max()

tensor(35.7825)

In [56]:
(stats_d_3["pred_bin_batch_idxs"] == stats_d_4["pred_bin_batch_idxs"]).all()

tensor(True)

In [57]:
(stats_d_3["pred_bin_idxs"] == stats_d_4["pred_bin_idxs"]).all()

tensor(True)